# Base Clients

> The core abstraction for different FL Clients. Any gneral client functionality resides here.

In [ ]:
#| default_exp clients

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import networkx as nx
import pickle
import json
from collections import defaultdict, OrderedDict
from copy import deepcopy
import random
from enum import Enum
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from peft import *
from community import community_louvain
from fedai.utils import *
from fedai.client_selector import *
from fedai.optimizers import *
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fedai.utils import *
from fedai.metrics import *
from fedai.losses import *
from transformers import AutoTokenizer
from omegaconf.dictconfig import DictConfig
import numpy as np
import math
np.random.seed(42)
torch.manual_seed(42)

<torch._C.Generator>

In [ ]:
#| export
class AgentRole(Enum):
    SERVER = 1
    CLIENT = 2
    MARL = 3

## BaseClient

This class implements FedAvg-like client and is considered the base class for all type of clients

A FL client is an entity that has a state and exist in an environment. In the case of Federated learning (FL), the agent's state is defined as its own model, data, criterion, optimizer. FL Focuses on distributed model training across multiple clients (agents), each with its local data. Clients **collaborate** to improve a global or shared model while keeping their data private. Communication is often periodic (e.g., every few training rounds). On the other hand, Multi-agent RL systems (MARL) Involves multiple agents interacting with an environment to learn policies for specific tasks (e.g., navigation, resource allocation). Each agent has a state also, but the state represntation might differ slightly from that of an FL agent. The data is often not preloaded as in FL rather, it's collected from the environemnt.

In [ ]:
#| export
class BaseClient:
    # A Federated Learning client implementing `FedAVG`.
    def __init__(self,
                 id, # the id of the agent
                 cfg, # the configuration of the agent.
                 state= None, # the state of the agent (model, optimizer, loss_fn), etc.
                 role= AgentRole.CLIENT, # the role of the agent (client or server)
                 block= None # The data block (local data of the FL Agent).
                 ):  
                 
        self.id = id 
        self.cfg = cfg 
        self.state = state
        self.role = role

        self.train_ds, self.test_ds = block[0], block[1]
        
        for key, value in self.state.items():
            setattr(self, key, value)

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
        self.train_loader = prepare_dl(self.cfg, self.train_ds)  # noqa: F405
        self.test_loader = prepare_dl(self.cfg, self.test_ds) # noqa: F405

        self.training_metrics = Metrics(list(self.cfg.training_metrics))  # noqa: F405
        self.test_metrics = Metrics(list(self.cfg.test_metrics))  # noqa: F405

        self.data_key, self.label_key = 'x', 'y'

In [ ]:
#| export 
@patch
def server_init(self: BaseClient, client_fn, client_selector, client_cls, loss_fn, writer):
    self.client_fn = client_fn
    self.client_selector = client_selector
    self.client_cls = client_cls
    self.loss_fn = loss_fn
    self.writer = writer
    self.latest_round = {}
    self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

Since data blocks are already on the disk, and since RL agents don't have a preloaded data blocks, we don't include the data in the FL agent's state. Another ratioanle behind this decision is that, state should contain dynamic objects that change over the interaction of the agents and data blocks are static in the case of FL agents, unless you are doing FL-RL Agents. 

For every client abstraction, whether it a base or any other type of federated client, it will initalize the training locally with a set of steps. This might include things like extracting the peft model out of the base model (in the case of LLMs clients). Also, it will terminate the local training with some steps, like saving the model state dictionary and so on.

In [ ]:
#| export
@patch
def runFL(self: BaseClient):
    res =  []
    all_ids = self.client_selector.select()
    
    for t in range(1, self.cfg.n_rounds + 1):
        lst_active_ids = all_ids[t-1]
        len_clients_ds = {}
        
        for id in lst_active_ids:
            client = self.client_fn(self.client_cls, self.cfg, id, self.latest_round, t, self.loss_fn, state_dir= self.cfg.state_dir)
            len_clients_ds[id] = len(client.train_ds)
            
            self.communicate(client) 
            client.fit()

            client.communicate(self) 
            self.latest_round[id] = t 

        self.aggregate(lst_active_ids, t, len_clients_ds)
        
        train_res, test_res = self.evaluate(t)
        train_df, test_df = self.writer.write(lst_active_ids, train_res, test_res, t) 
        res.append((train_df, test_df))
        
    self.writer.save(res)
    self.writer.finish()

    return res

### Helpers

We will adjust the string reprsntation of the client abstraction to make it more meaningful.

In [ ]:
#| export
@patch
def __str__(self: BaseClient) -> str:
    return f'''{self.__class__.__name__}: {self.__class__.__name__}
    Index : {self.id}
    Model: {self.model.__class__.__name__}
    Criterion: {self.criterion.__class__.__name__}
    Optimizer: {self.optimizer.__class__.__name__}'''


In [ ]:
#| export
@patch
def clear_model(self: BaseClient):
    self.model = None if hasattr(self, 'model') else None

In [ ]:
#| export
@patch
def update_parameters(self:BaseClient, new_params):
    # inplace update of the model parameters with new_params.
    if not hasattr(self, 'model'):
        raise ValueError("Model is not initialized. Please initialize the model before updating parameters.")
    
    with torch.no_grad():
        for param , new_param in zip(self.model.parameters(), new_params):
            param.copy_(new_param.data)


In [ ]:
#| export
@patch
def send_to_device(self: BaseClient, batch):
    return {k: v.to(self.device) for k, v in batch.items()}

### Training

In [ ]:
#| export
@patch
def _forward(self: BaseClient, batch):
    X, y = batch['x'], batch['y']
    outputs = self.model(X)
    loss = self.criterion(outputs, y)
    return loss, outputs

In [ ]:
#| export
@patch
def _closure(self: BaseClient, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    try:
        loss, logits = self._forward(batch)
        probs = torch.nn.functional.softmax(logits, dim=-1)
        y_pred = probs.argmax(dim=-1)
        y_true = batch[self.label_key]

        if hasattr(self, "training_metrics") and self.cfg.training_metrics:
            if hasattr(self, "tokenizer"):
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true, tokenizer=self.tokenizer)
            else:
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true)

    except Exception as e:
        print(f"Error in _closure: {e}")
        return torch.tensor(0.0, dtype=torch.float32, device=self.device), metrics  # Return safe values

    return loss, metrics


In [ ]:
#| export
@patch
def _run_batch(self: BaseClient, batch: dict) -> tuple:
    self.optimizer.zero_grad()
    loss, metrics = self._closure(batch)

    if loss.item() == 0.0:
        return loss, metrics
    
    loss.backward()
    
    if self.cfg.model.grad_norm_clip:
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.model.grad_norm_clip)

    self.optimizer.step()

    return loss, metrics

In [ ]:
#| export
@patch
def _run_epoch(self: BaseClient):

    for i, batch in enumerate(self.train_loader):
        batch = self.send_to_device(batch)
        self._run_batch(batch)

In [ ]:
#| export
@patch
def fit(self: BaseClient) -> dict:
    
    self.model.to(self.device)
    self.model.train()
    for _ in range(self.cfg.local_epochs):
        self._run_epoch()


### Evaluation




In [ ]:
#| export
@patch
def train_test_stats(self: BaseClient, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    try:
        X, y = batch['x'], batch['y']
        logits = self.model(X)
        loss = self.criterion(logits, y)
        probs = torch.nn.functional.softmax(logits, dim=-1)
        y_pred = probs.argmax(dim=-1)
        y_true = batch[self.label_key]

        if hasattr(self, "training_metrics") and self.cfg.training_metrics:
            if hasattr(self, "tokenizer"):
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true, tokenizer=self.tokenizer)
            else:
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true)

    except Exception as e:
        return torch.tensor(0.0, dtype=torch.float32, device=self.device), metrics  # Return safe values

    return loss, metrics


In [ ]:
#| export
@patch
def evaluate_local(self: BaseClient, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []

    self.model = self.model.to(self.device)
    self.model.eval()
    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.train_test_stats(batch)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    return {"loss": avg_loss, "metrics": total_metrics}


In [ ]:
#| export
@patch
def evaluate(self: BaseClient, t):
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):
        client = self.client_fn(self.client_cls, self.cfg, id, self.latest_round, t, self.loss_fn, state_dir= self.cfg.state_dir)
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
    return lst_train_res, lst_test_res    


> To do: implement the communication process in **Protobuf**.

### Communication

Communication refers to the process of downloading and uploading models from the server and to the client. Since we are safeguarding against memory issues, we use sequential client processing and disk checkpointing as our tools.

In [ ]:
#| export
@patch
def save_state(self: BaseClient, state_dict):  # noqa: F811
    # save the model to self.cfg.save_dir/comm_round/f"local_output_{id}"/state.pth
    
    state_path = os.path.join(self.cfg.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
    if not os.path.exists(os.path.dirname(state_path)):
        os.makedirs(os.path.dirname(state_path))

    state_dict['model'] = self.model.state_dict()
    state_dict['optimizer'] = self.optimizer.state_dict()

    torch.save(state_dict, state_path)

    if self.role == AgentRole.CLIENT:
        save_space(self)


In [ ]:
#| export
@patch
def communicate(self: BaseClient, another_agent: Agent):  # noqa: F811
    if self.role == AgentRole.CLIENT:
        self.save_state(self.state)

### Aggregation


The aggegation for `fedavg` is defined as:

$$m_t \leftarrow \sum_{k \in S_t} n_k$$
$$W_{global}^{(t + 1)} \leftarrow \sum_{k \in S_t} \frac{n_k}{m_t} w_k^{(t + 1)}$$

where $S_t$ is the set of active clients at communication round $t$.

In [ ]:
#| export
@patch
def aggregate(self: BaseClient, lst_active_ids, comm_round, len_clients_ds):
        
    m_t = sum(len_clients_ds.values())
    with torch.no_grad():
        for i, id in enumerate(lst_active_ids):
            state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
            
            state = torch.load(state_path, weights_only=False)
            client_state_dict = state['model']

            if i == 0:
                global_model = {
                    key: torch.zeros_like(value) 
                    for key, value in client_state_dict.items()
                }

            n_k = len_clients_ds[id]
            weight =  n_k / m_t 

        
            for key in client_state_dict.keys():
                global_model[key].add_(weight * client_state_dict[key])


        server_state = {
            'model': global_model,
        }

        server_state_path = os.path.join(self.cfg.save_dir, str(comm_round), "global_model", "state.pth")
        
        if not os.path.exists(os.path.dirname(server_state_path)):
            os.makedirs(os.path.dirname(server_state_path), exist_ok=True)
            
        torch.save(server_state, server_state_path)

    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()